In [1]:
!pip install git+https://github.com/openai/CLIP.git ftfy regex tqdm

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-skuqz7hv
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-skuqz7hv
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.5 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=b8e0b497b768c922f4e82def0d32ae2b2bdc755d88ffa0709e1f2ba30323c541
  Stored in directory: /tmp/pip-ephem-wheel-cache-1ri2nqr4/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip


In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import clip
import numpy as np
import pandas as pd
import html
import random
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import accuracy_score

In [3]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

DATA_DIR = "/kaggle/input/datasets/gianmarco96/upmcfood101"
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "images/train")
TEST_IMG_DIR = os.path.join(DATA_DIR, "images/test")

TRAIN_TEXT = os.path.join(DATA_DIR, "texts/train_titles.csv")
TEST_TEXT = os.path.join(DATA_DIR, "texts/test_titles.csv")

# Load pre-trained CLIP model
model, preprocess = clip.load("ViT-B/32", device=DEVICE)
model = model.float()

Using device: cuda


100%|████████████████████████████████████████| 338M/338M [00:01<00:00, 307MiB/s]


In [4]:
train_df = pd.read_csv(TRAIN_TEXT, header=None)
test_df = pd.read_csv(TEST_TEXT, header=None)

train_df.columns = ["image_name", "text", "class"]
test_df.columns = ["image_name", "text", "class"]

def build_text_map(df):
    mapping = {}
    for _, row in df.iterrows():
        img = row["image_name"]
        text = html.unescape(str(row["text"]))
        cls = row["class"]
        full_text = f"{text}"
        mapping[img] = full_text
    return mapping

train_text_map = build_text_map(train_df)
test_text_map = build_text_map(test_df)

In [5]:
def load_image_paths(root_dir):
    image_paths = []
    labels = []
    class_names = sorted(os.listdir(root_dir))
    class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

    for cls in class_names:
        cls_folder = os.path.join(root_dir, cls)
        for img_name in os.listdir(cls_folder):
            image_paths.append(os.path.join(cls_folder, img_name))
            labels.append(class_to_idx[cls])

    return image_paths, labels, class_names

train_paths, train_labels, class_names = load_image_paths(TRAIN_IMG_DIR)
test_paths, test_labels, _ = load_image_paths(TEST_IMG_DIR)

def sample_k_shot(paths, labels, k=5):
    from collections import defaultdict
    data = defaultdict(list)
    for p, l in zip(paths, labels):
        data[l].append(p)

    new_paths = []
    new_labels = []
    for l, items in data.items():
        sampled = random.sample(items, min(k, len(items)))
        new_paths.extend(sampled)
        new_labels.extend([l]*len(sampled))

    return new_paths, new_labels

few_train_paths, few_train_labels = sample_k_shot(train_paths, train_labels, k=5)

In [6]:
def multimodal_zero_shot(test_paths, test_labels, class_names, text_map):
    print("Bắt đầu đánh giá Zero-shot...")
    prompts = [f"a photo of {c}" for c in class_names]
    text_tokens = clip.tokenize(prompts).to(DEVICE)

    with torch.no_grad():
        class_features = model.encode_text(text_tokens)
        class_features /= class_features.norm(dim=-1, keepdim=True)

    preds = []
    for path in tqdm(test_paths, desc="Zero-shot Progress"):
        img_name = os.path.basename(path)
        real_text = text_map.get(img_name, "")
        image = preprocess(Image.open(path)).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            image_feat = model.encode_image(image)
            image_feat /= image_feat.norm(dim=-1, keepdim=True)

            text_tokens = clip.tokenize([real_text], truncate=True).to(DEVICE)
            text_feat = model.encode_text(text_tokens)
            text_feat /= text_feat.norm(dim=-1, keepdim=True)

            sim_class = image_feat @ class_features.T
            sim_text = image_feat @ text_feat.T

            combined = sim_class + 0.3 * sim_text
            pred = combined.argmax().item()

        preds.append(pred)

    acc = accuracy_score(test_labels, preds)
    print(f"\\nMultimodal Zero-shot Accuracy: {acc:.4f}")
    return acc

zs_acc = multimodal_zero_shot(test_paths, test_labels, class_names, test_text_map)

Bắt đầu đánh giá Zero-shot...


Zero-shot Progress: 100%|██████████| 22716/22716 [11:39<00:00, 32.46it/s]


\nMultimodal Zero-shot Accuracy: 0.5908


In [7]:
class FoodMultimodalDataset(Dataset):
    def __init__(self, paths, labels, text_map, preprocess):
        self.paths = paths
        self.labels = labels
        self.text_map = text_map
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]
        img_name = os.path.basename(path)
        text = self.text_map.get(img_name, "")

        image = self.preprocess(Image.open(path).convert("RGB"))
        text_tokens = clip.tokenize([text], truncate=True).squeeze(0)

        return image, text_tokens, label

def extract_features_batched(paths, labels, text_map, batch_size=128):
    dataset = FoodMultimodalDataset(paths, labels, text_map, preprocess)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    all_features = []
    all_labels = []

    model.eval()
    with torch.no_grad():
        for images, texts, lbls in tqdm(loader, desc="Extracting Features"):
            images = images.to(DEVICE)
            texts = texts.to(DEVICE)

            img_feat = model.encode_image(images)
            img_feat /= img_feat.norm(dim=-1, keepdim=True)

            txt_feat = model.encode_text(texts)
            txt_feat /= txt_feat.norm(dim=-1, keepdim=True)

            combined = torch.cat([img_feat, txt_feat], dim=-1)
            
            all_features.append(combined.cpu())
            all_labels.append(lbls)

    return torch.cat(all_features).numpy(), torch.cat(all_labels).numpy()

In [8]:
class LinearProbingClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.fc(x)

def train_pytorch_classifier(X_train, y_train, X_test, y_test, num_classes=101, epochs=15, batch_size=256, lr=1e-3):
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

    input_dim = X_train.shape[1]
    classifier = LinearProbingClassifier(input_dim, num_classes).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(classifier.parameters(), lr=lr, weight_decay=1e-4)

    best_acc = 0.0

    print(f"Đang huấn luyện với {len(X_train)} mẫu...")
    for epoch in range(epochs):
        classifier.train()
        total_loss = 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(DEVICE), batch_y.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = classifier(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()

        classifier.eval()
        all_preds = []
        all_trues = []
        
        with torch.no_grad():
            for batch_x, batch_y in test_loader:
                batch_x = batch_x.to(DEVICE)
                outputs = classifier(batch_x)
                preds = outputs.argmax(dim=1).cpu().numpy()
                all_preds.extend(preds)
                all_trues.extend(batch_y.numpy())
                
        acc = accuracy_score(all_trues, all_preds)
        if acc > best_acc:
            best_acc = acc
            
        if (epoch + 1) % 5 == 0 or epoch == epochs - 1:
            print(f"Epoch [{epoch+1}/{epochs}] - Loss: {total_loss/len(train_loader):.4f} - Test Acc: {acc:.4f}")

    return best_acc

In [9]:
def extract_multimodal_features(paths, text_map):
    feats = []

    for path in tqdm(paths):
        img_name = os.path.basename(path)
        text = text_map.get(img_name, "")

        image = preprocess(Image.open(path)).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            img_feat = model.encode_image(image)
            img_feat /= img_feat.norm(dim=-1, keepdim=True)

            txt_feat = model.encode_text(clip.tokenize([text], truncate=True).to(DEVICE))
            txt_feat /= txt_feat.norm(dim=-1, keepdim=True)

        combined = torch.cat([img_feat, txt_feat], dim=-1)
        feats.append(combined.cpu().numpy()[0])

    return np.array(feats)

In [10]:
print("--- BƯỚC 1: TRÍCH XUẤT ĐẶC TRƯNG ---")
print("1. Trích xuất tập TEST...")
X_test, y_test = extract_features_batched(test_paths, test_labels, test_text_map)

print("\\n2. Trích xuất tập FEW-SHOT TRAIN...")
X_train_few, y_train_few = extract_features_batched(few_train_paths, few_train_labels, train_text_map)

print("\\n3. Trích xuất TOÀN BỘ TẬP TRAIN...")
X_train_full, y_train_full = extract_features_batched(train_paths, train_labels, train_text_map)

print("\\n" + "="*50)
print("=== TỔNG HỢP KẾT QUẢ ===")
print("="*50)

print(f"Kịch bản 1 (Zero-shot) Accuracy : {zs_acc:.4f}")

print(f"\\n--- Kịch bản 2: Few-shot (k=5) ---")
few_shot_acc = train_pytorch_classifier(X_train_few, y_train_few, X_test, y_test, epochs=30, batch_size=32)
print(f"-> Tốt nhất Few-shot Accuracy: {few_shot_acc:.4f}")

print(f"\\n--- Kịch bản 3: Full Train ---")
full_train_acc = train_pytorch_classifier(X_train_full, y_train_full, X_test, y_test, epochs=20, batch_size=256)
print(f"-> Tốt nhất Full Train Accuracy: {full_train_acc:.4f}")

--- BƯỚC 1: TRÍCH XUẤT ĐẶC TRƯNG ---
1. Trích xuất tập TEST...


Extracting Features: 100%|██████████| 178/178 [02:17<00:00,  1.30it/s]


\n2. Trích xuất tập FEW-SHOT TRAIN...


Extracting Features: 100%|██████████| 4/4 [00:05<00:00,  1.50s/it]


\n3. Trích xuất TOÀN BỘ TẬP TRAIN...


Extracting Features: 100%|██████████| 532/532 [10:01<00:00,  1.13s/it]


\n==================================================
=== TỔNG HỢP KẾT QUẢ ===
Kịch bản 1 (Zero-shot) Accuracy : 0.5908
\n--- Kịch bản 2: Few-shot (k=5) ---
Đang huấn luyện với 505 mẫu...
Epoch [5/30] - Loss: 4.2436 - Test Acc: 0.7401
Epoch [10/30] - Loss: 3.8280 - Test Acc: 0.7972
Epoch [15/30] - Loss: 3.4310 - Test Acc: 0.8131
Epoch [20/30] - Loss: 3.0558 - Test Acc: 0.8218
Epoch [25/30] - Loss: 2.7087 - Test Acc: 0.8246
Epoch [30/30] - Loss: 2.3804 - Test Acc: 0.8285
-> Tốt nhất Few-shot Accuracy: 0.8285
\n--- Kịch bản 3: Full Train ---
Đang huấn luyện với 67988 mẫu...
Epoch [5/20] - Loss: 0.9861 - Test Acc: 0.9053
Epoch [10/20] - Loss: 0.8153 - Test Acc: 0.9091
Epoch [15/20] - Loss: 0.7958 - Test Acc: 0.9102
Epoch [20/20] - Loss: 0.7901 - Test Acc: 0.9088
-> Tốt nhất Full Train Accuracy: 0.9102
